This implements advisory agent

In [182]:
import investor_profiles
import stock_universe
import disclosure_snippets
import numpy as np
import json
import array

In [183]:
RISK_PROFILES = [
{"risk_tolerance": "Conservative", "distribution_strategy": "equal-weight", "stocks_list": ["PAYBOND", "PAYGOLD", "PAYRETAIL"]},
{"risk_tolerance": "Moderate", "distribution_strategy": "equal-weight", "stocks_list": ["PAYRETAIL", "PAYINFRA", "PAYGOLD"]},
{"risk_tolerance": "Aggressive", "distribution_strategy": "equal-weight", "stocks_list": ["PAYTECH", "PAYFIN", "PAYINFRA"]}
]

In [184]:
# Defines the tools

def get_stock_data(ticker: str):
  return stock_universe.STOCK_UNIVERSE.get(ticker), stock_universe.MARKET_RETURN, stock_universe.RISK_FREE_RATE

def calculate_return_stock(Rm: float, Rf: float, beta: float):
  return Rf + beta * (Rm - Rf)

def calculate_return_portfolio(weights: np.array, stocks: np.array, Rm: float, Rf: float, correlation: float):
  weighted_return = 0
  for i in range(len(stocks)):
    #print(stocks[i])
    weighted_return += calculate_return_stock(Rm, Rf, stocks[i].get('beta')) * weights[i]
    #print("Current weighted return: ", weighted_return)
  return weighted_return

def calculate_variance_portfolio(weights: np.array, stocks: np.array, correlation: float):
  # Var(R_p) = Σᵢ wᵢ²σᵢ² + 2·Σ_{i<j} wᵢwⱼ·Cov(Rᵢ,Rⱼ), with Cov(Rᵢ,Rⱼ) = ρ·σᵢ·σⱼ
  # correlation ρ = 0.3 for every pair of the three tickers. Convert variance to portfolio standard deviation
  variance = 0
  # Calculate Σᵢ wᵢ²σᵢ²
  for i in range(len(stocks)):
    sigma_i = stocks[i].get('std_dev')
    variance += (weights[i]**2) * (sigma_i**2)

  # Calculate 2·Σ_{i<j} wᵢwⱼ·Cov(Rᵢ,Rⱼ)
  for i in range(len(stocks)):
    for j in range(i + 1, len(stocks)):
      sigma_i = stocks[i].get('std_dev')
      sigma_j = stocks[j].get('std_dev')
      covariance = correlation * sigma_i * sigma_j
      variance += 2 * weights[i] * weights[j] * covariance
      #print("Current variance: ", variance)
  return np.sqrt(variance)

def create_stocks_portfolio(risk_tolerance: str):
  weights = []
  stocks_arr = []
  for risk in RISK_PROFILES:
    if risk_tolerance == risk.get("risk_tolerance"):
      distribution_strategy = risk.get("distribution_strategy")
      match distribution_strategy:
          case "equal-weight":
            stocks = risk.get("stocks_list")
            number_stocks = len(stocks)
            for i in range(0, number_stocks):
              stock_data, Rm, Rf = get_stock_data(stocks[i])
              weights.append( 1 / number_stocks)
              stocks_arr.append(stock_data)
            return weights, stocks_arr, Rm, Rf, 0.3
          case _:
            return "Invalid data"

def escalate_human(volatility: float):
  return volatility > 0.20

def run_tool(name, args):
    """Dispatch a tool call and print the think/act/observe trace."""
    print(f"   ACTION     -> {name}({json.dumps(args)})")
    result = TOOLS[name](**args)
    # Convert numpy boolean to Python boolean if necessary for JSON serialization
    if isinstance(result, (np.bool_)):
        result = bool(result)
    print(f"   OBSERVATION <- {json.dumps(result)}\n")
    return result

In [185]:
TOOLS = {
    "create_stocks_portfolio": create_stocks_portfolio,
    "get_stock_data": get_stock_data,
    "calculate_return_stock": calculate_return_stock,
    "calculate_return_portfolio": calculate_return_portfolio,
    "calculate_variance_portfolio": calculate_variance_portfolio,
    "escalate_human": escalate_human
}

# Tool schemas the LLM can read (only used in live mode)
TOOL_SCHEMAS = [
    {"name": "get_stock_data", "description": "Get details like beta, analyst expected return std_dev for a given stock identified by its ticker",
     "input_schema": {"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]}},
    {"name": "calculate_return_stock", "description": "Calculate returns as per CAPM formula, using Rf (risk free return), Rm (Market return), beta (Beta for the stock)",
     "input_schema": {"type": "object", "properties": {"Rm": {"type": "float"}, "Rf": {"type": "float"}, "beta": {"type": "float"}}, "required": ["Rm", "Rf", "beta"]}},
    {"name": "calculate_return_portfolio", "description": "Calculate return for the portfolio using weights (array of weights of stocks in portfolio), "
    "stocks (array of stocks in portfolio), Rm (Market return), Rf (Risk free return), correlation (correlation between stocks)",
     "input_schema": {"type": "object", "properties": {"weights": {"type": "array of float"}, "stocks": {"type": "array of float"}, "Rm": {"type": "float"}, "Rf": {"type": "float"},
                                                       "correlation": {"type": "float"}}, "required": ["weights", "stocks", "Rm", "Rf", "correlation"]}},
    {"name": "calculate_variance_portfolio", "description": "Calculate Standard deviation for the portfolio using weights (array of weights of stocks in portfolio), "
    "stocks (array of stocks in portfolio), correlation (correlation between stocks)",
     "input_schema": {"type": "object", "properties": {"weights": {"type": "array of float"}, "stocks": {"type": "array of float"}, "correlation": {"type": "float"}},
                      "required": ["weights", "stocks", "Rm", "Rf", "correlation"]}}
    ]

# Steps to follow in scripted mode
AGENT_STEPS = [
    "create_stocks_portfolio",

]


In [186]:
stock_data, Rm, Rf = get_stock_data("PAYBOND")
#print("Stock Data -> ", Rm, Rf, stock_data)
#print("Return for stock : ", f"{calculate_return_stock(Rm, Rf, stock_data.get('beta')):.2}")
weights = [1/3, 1/3, 1/3]
stocks= []
stock_data, Rm, Rf = get_stock_data("PAYBOND")
stocks.append(stock_data)
stock_data, Rm, Rf = get_stock_data("PAYGOLD")
stocks.append(stock_data)
stock_data, Rm, Rf = get_stock_data("PAYRETAIL")
stocks.append(stock_data)
portfolio_return = calculate_return_portfolio(weights, stocks, Rm, Rf, 0.3)
portfolio_std_dev = calculate_variance_portfolio(weights=weights, stocks=stocks, correlation=0.3)
print("Portfolio return: ", f"{portfolio_return:.2}")
print("Portfolio variance: ", f"{portfolio_std_dev:.2}")

Portfolio return:  0.092
Portfolio variance:  0.084


Build the definitive pipeline below

In [188]:
#print(create_stocks_portfolio("Conservative"))

# Step 1, Read Investor profile record one by one from investor_profiles.py file
for investor in investor_profiles.INVESTOR_PROFILES:
  print("Investor id : ", investor.get("investor_id"), investor.get("risk_tolerance"))

# Step 2, Call create_stocks_portfolio passing each investor's risk_tolerance. It internally calls get_stock_data
# and returns weights, stocks_arr, Rm, Rf
  inv_weights, inv_stocks, Rm, Rf, correlation = run_tool("create_stocks_portfolio", {"risk_tolerance": investor.get("risk_tolerance")})

# Step 3, Call calculate_return_portfolio passing weights, stocks, Rm, Rf, correlation
  result_return = run_tool("calculate_return_portfolio", {"weights": inv_weights, "stocks": inv_stocks, "Rm": Rm, "Rf": Rf, "correlation": correlation })

# Step 4, Call calculate_variance_portfolio passing  weights, stocks, correlation
  result_variance = run_tool("calculate_variance_portfolio", {"weights": inv_weights, "stocks": inv_stocks, "correlation": correlation})

# Step 5, Escalate to human if volatility > 20
  escalation = run_tool("escalate_human", {"volatility": result_variance})
# Print the output
  if not escalation:
    for risk in RISK_PROFILES:
      if investor.get("risk_tolerance") == risk.get("risk_tolerance"):
        print(f"For {investor.get("risk_tolerance")} investor {investor.get("investor_id")}, we recommend an allocation across {risk.get("stocks_list")} with an expected portfolio return of {result_return:.1%} and volatility of {result_variance:.1%}")
  else:
     print(f"ESCALATED_TO_HUMAN_ADVISOR, volatility: {result_variance:.1%}")
  print("-" * 150)

Investor id :  INV01 Conservative
   ACTION     -> create_stocks_portfolio({"risk_tolerance": "Conservative"})
   OBSERVATION <- [[0.3333333333333333, 0.3333333333333333, 0.3333333333333333], [{"beta": 0.05, "analyst_expected_return": 0.065, "std_dev": 0.04}, {"beta": 0.2, "analyst_expected_return": 0.08, "std_dev": 0.12}, {"beta": 0.85, "analyst_expected_return": 0.11, "std_dev": 0.17}], 0.13, 0.07, 0.3]

   ACTION     -> calculate_return_portfolio({"weights": [0.3333333333333333, 0.3333333333333333, 0.3333333333333333], "stocks": [{"beta": 0.05, "analyst_expected_return": 0.065, "std_dev": 0.04}, {"beta": 0.2, "analyst_expected_return": 0.08, "std_dev": 0.12}, {"beta": 0.85, "analyst_expected_return": 0.11, "std_dev": 0.17}], "Rm": 0.13, "Rf": 0.07, "correlation": 0.3})
   OBSERVATION <- 0.092

   ACTION     -> calculate_variance_portfolio({"weights": [0.3333333333333333, 0.3333333333333333, 0.3333333333333333], "stocks": [{"beta": 0.05, "analyst_expected_return": 0.065, "std_dev": 0

In [133]:
class InvestorProfile:
  def __init__(self, investor_id: str, risk_tolerance: str, horizon_years: int, investment_amount_inr: int):
    self.investor_id = investor_id
    self.risk_tolerance = risk_tolerance
    self.horizon_years = horizon_years
    self.investment_amount_inr = investment_amount_inr

def portfolio_agent_scripted(InvestorProfile_obj):
  print(TOOL_SCHEMAS[0].get("name"))
  print(InvestorProfile_obj.risk_tolerance, InvestorProfile_obj.horizon_years)
  # Follow the steps in TOOLS_SCHEMA in order
  for tool in TOOL_SCHEMAS:
    tool_name = tool.get("name")
    print("Tool to invoke --> ", tool_name)

investor = InvestorProfile("INV01", "Conservative", 3, 200000)
portfolio_agent_scripted(investor)

print(create_stocks_portfolio("Conservative"))

get_stock_data
Conservative 3
Tool to invoke -->  get_stock_data
Tool to invoke -->  calculate_return_stock
Tool to invoke -->  calculate_return_portfolio
Tool to invoke -->  calculate_variance_portfolio
([0.3333333333333333, 0.3333333333333333, 0.3333333333333333], [{'beta': 0.05, 'analyst_expected_return': 0.065, 'std_dev': 0.04}, {'beta': 0.2, 'analyst_expected_return': 0.08, 'std_dev': 0.12}, {'beta': 0.85, 'analyst_expected_return': 0.11, 'std_dev': 0.17}], 0.13, 0.07, 0.3)
